# 11.2 - Keyword Search

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Before semantic (vector) search we need the baseline: word-based retrieval that matches exact terms. TF-IDF and BM25 rank documents by how well their words match the query's words. This is fast, interpretable, needs no embeddings, and beats vector search on exact-match queries (product codes, model numbers, names).

## 2. Why Does This Matter?

Keyword search underlies every production retrieval system. Elasticsearch defaults to BM25. Hybrid systems keep a keyword engine alongside the vector engine - so you must understand the baseline to know when it wins.

## 3. Prerequisites

Phase 09 (GenAI), Phase 10 (LLMs), basic Python.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build TF-IDF and BM25 from scratch with numpy (rank_bm25 is NOT installed)
- Explain term frequency, inverse document frequency, and length normalization
- Build an inverted index and use it for ranked retrieval
- Apply preprocessing (lowercase + stopword removal)

## 5. Mental Model

Keyword search is a librarian who finds books by matching words in the title/index: fast and reliable for exact terms, but it misses synonyms and context.

```text
Index:  { term: [(doc_id, tf, doc_len), ...] }   (inverted index)
Query:  tokenize -> score each doc -> rank -> top-k
```


## 6. Setup: a Small FAQ Corpus + Preprocessing
First we tokenize (lowercase) and remove stop words, which are frequent words with little retrieval value (the 'not' caveat notwithstanding).

In [1]:
import numpy as np

CORPUS = [
    "The return policy allows returns within 30 days of purchase.",
    "Shipping takes 5 to 7 business days for standard delivery.",
    "Returns are accepted only for unopened items in original packaging.",
    "Refunds are processed within 5 to 7 business days after we receive the item.",
    "Express shipping costs extra and arrives in 2 to 3 business days.",
]

STOPWORDS = {"the", "a", "an", "to", "for", "of", "in", "within", "and", "or", "is", "are"}


def tokenize(text):
    words = text.lower().replace(".", " ").replace(",", " ").split()
    return [w for w in words if w not in STOPWORDS]


tokens = [tokenize(d) for d in CORPUS]
for i, t in enumerate(tokens):
    print(i, t)


0 ['return', 'policy', 'allows', 'returns', '30', 'days', 'purchase']
1 ['shipping', 'takes', '5', '7', 'business', 'days', 'standard', 'delivery']
2 ['returns', 'accepted', 'only', 'unopened', 'items', 'original', 'packaging']
3 ['refunds', 'processed', '5', '7', 'business', 'days', 'after', 'we', 'receive', 'item']
4 ['express', 'shipping', 'costs', 'extra', 'arrives', '2', '3', 'business', 'days']


## 7. Term Frequency (TF)
How often a word appears in a document. A documents that uses a term more often is more relevant - but raw counts favour long documents.

In [2]:
def build_vocab(tokens):
    vocab = {}
    for doc_toks in tokens:
        for t in doc_toks:
            if t not in vocab:
                vocab[t] = len(vocab)
    return vocab


def tf_matrix(tokens, vocab):
    # TF value: 1 + log(count) if count > 0 else 0  (classic BM25-style smoothing)
    m = np.zeros((len(tokens), len(vocab)))
    for d, doc_toks in enumerate(tokens):
        for t in doc_toks:
            c = doc_toks.count(t)
            m[d, vocab[t]] = 1 + np.log(c)
    return m


vocab = build_vocab(tokens)
tf = tf_matrix(tokens, vocab)
print("vocab size:", len(vocab))
print("TF matrix shape (docs x vocab):", tf.shape)
print("TF of 'returns' across docs:", tf[:, vocab["returns"]])


vocab size: 32
TF matrix shape (docs x vocab): (5, 32)
TF of 'returns' across docs: [1. 0. 1. 0. 0.]


## 8. Inverse Document Frequency (IDF)
How rare a word is across all documents. Common words are discounted; rare, discriminating words are boosted (`idf = log(1 + N/(1+df))`).

In [3]:
def idf_vector(tokens, vocab):
    n_docs = len(tokens)
    idf = np.zeros(len(vocab))
    for term, col in vocab.items():
        df = sum(1 for t in tokens if term in t)
        idf[col] = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))
    return idf


idf = idf_vector(tokens, vocab)
for term in ["returns", "express", "shipping", "packaging"]:
    print(f"idf({term:10s}) = {idf[vocab[term]]:.3f}")


idf(returns   ) = 0.875
idf(express   ) = 1.386
idf(shipping  ) = 0.875
idf(packaging ) = 1.386


## 9. TF-IDF + Inverted Index Scoring
TF-IDF score of a term in a doc = TF x IDF. We score a query by summing the TF-IDF weight of its terms. An inverted index maps each term to the docs containing it so we only score docs that share a term with the query.

In [4]:
def tfidf_scores(query, tokens, tf, idf, vocab):
    q_toks = tokenize(query)
    scores = np.zeros(len(tokens))
    inverted = {t: [d for d, dt in enumerate(tokens) if t in dt] for t in vocab}
    for t in q_toks:
        if t not in vocab:
            continue
        for d in inverted[t]:
            scores[d] += tf[d, vocab[t]] * idf[vocab[t]]
    return q_toks, scores


q = "return policy with packaging"
_, scores = tfidf_scores(q, tokens, tf, idf, vocab)
order = np.argsort(-scores)
for d in order:
    print(f"doc{d}: {scores[d]:.3f} | {CORPUS[d]}")


doc0: 2.773 | The return policy allows returns within 30 days of purchase.
doc2: 1.386 | Returns are accepted only for unopened items in original packaging.
doc1: 0.000 | Shipping takes 5 to 7 business days for standard delivery.
doc3: 0.000 | Refunds are processed within 5 to 7 business days after we receive the item.
doc4: 0.000 | Express shipping costs extra and arrives in 2 to 3 business days.


## 10. BM25 from Scratch
BM25 improves on TF-IDF with **length normalization**: terms in a short document are worth more than the same count in a long one. It is the default in Elasticsearch.

`score(d,q) = sum over query terms of idf(t) * tf(t,d)*(k1+1) / (tf(t,d) + k1*(1 - b + b*len(d)/avg_len))`

In [5]:
def bm25_scores(query, tokens, vocab, k1=1.5, b=0.75):
    n_docs = len(tokens)
    avg_len = np.mean([len(d) for d in tokens])
    idf = idf_vector(tokens, vocab)
    term_df = {t: sum(1 for d in tokens if t in d) for t in vocab}
    scores = np.zeros(n_docs)
    q_toks = tokenize(query)
    for t in q_toks:
        if t not in vocab:
            continue
        w = idf[vocab[t]]
        for d, dt in enumerate(tokens):
            if t not in dt:
                continue
            tf_t = dt.count(t)
            ratio = len(dt) / avg_len
            denom = tf_t + k1 * (1 - b + b * ratio)
            scores[d] += w * (tf_t * (k1 + 1)) / denom
    return scores


for q in ["refund time", "express shipping cost", "return unopened item"]:
    s = bm25_scores(q, tokens, vocab)
    order = np.argsort(-s)
    print(f"QUERY: {q!r}")
    for d in order:
        print(f"   doc{d}: {s[d]:.4f} | {CORPUS[d][:48]}")
    print()


QUERY: 'refund time'
   doc0: 0.0000 | The return policy allows returns within 30 days 
   doc1: 0.0000 | Shipping takes 5 to 7 business days for standard
   doc2: 0.0000 | Returns are accepted only for unopened items in 
   doc3: 0.0000 | Refunds are processed within 5 to 7 business day
   doc4: 0.0000 | Express shipping costs extra and arrives in 2 to

QUERY: 'express shipping cost'
   doc4: 2.1666 | Express shipping costs extra and arrives in 2 to
   doc1: 0.8852 | Shipping takes 5 to 7 business days for standard
   doc2: 0.0000 | Returns are accepted only for unopened items in 
   doc0: 0.0000 | The return policy allows returns within 30 days 
   doc3: 0.0000 | Refunds are processed within 5 to 7 business day

QUERY: 'return unopened item'
   doc0: 1.4840 | The return policy allows returns within 30 days 
   doc2: 1.4840 | Returns are accepted only for unopened items in 
   doc3: 1.2617 | Refunds are processed within 5 to 7 business day
   doc1: 0.0000 | Shipping takes 5 to 7 busin

## 11. TF-IDF vs BM25 on the Corpus
Both rank the same FAQ doc top on an exact term. BM25's length normalization is why it usually generalises better - but for this tiny demo the ranking is similar.

In [6]:
q = "shipping takes business days"
_, s_tfidf = tfidf_scores(q, tokens, tf, idf, vocab)
s_bm25 = bm25_scores(q, tokens, vocab)
print(f"{'doc':4s} {'TF-IDF':>8s} {'BM25':>8s}")
for d in range(len(tokens)):
    print(f"{d:<4d} {s_tfidf[d]:8.3f} {s_bm25[d]:8.3f}")


doc    TF-IDF     BM25
0       0.288    0.308
1       3.088    3.123
2       0.000    0.000
3       0.827    0.752
4       1.702    1.631



## Common Mistakes

- Not handling case sensitivity or punctuation.
- Ignoring stop words when they matter (e.g. the word "not").
- Using raw term frequency without length normalization.
- Not normalizing text (lowercase, punctuation handling) consistently.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Expected doc not retrieved | Query terms don't match doc tokens | Print tokenized query and doc; add preprocessing |
| Rare words score too high | Rare query term matches noise | Check IDF; add stopword removal |
| Scores all near zero | Query too short / all stop words | Expand query or phrase match |
| Long docs always win | No length normalization | Use BM25 with b>0 |

## Best Practices

- Always lowercase and tokenize consistently.
- Prefer BM25 over raw TF-IDF.
- Include a keyword engine in hybrid retrieval systems.
- Benchmark on real queries before switching to vector search.
- Log retrieval scores for debugging.

## Hands-On Practice

1. **Basic:** Build a BM25 search over 10 documents.
2. **Guided:** Add preprocessing (lowercase, stopword removal) and compare.
3. **Independent:** Search an FAQ set and measure precision@3.
4. **Realistic:** Compare BM25 with cosine similarity on embeddings for the same corpus.
5. **Challenge:** Implement phrase matching ("return policy" as an exact phrase).

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
